<div dir="rtl" align="right" style="line-height: 1.8;">

# 🚀 بنچمارک و تحلیل معماری‌های LeNet-5: تقابل کلاسیک و مدرن

در این نوت‌بوک، یک آزمایش کنترل‌شده و دقیق برای بررسی رفتار معماری‌های شبکه‌های عصبی پیچشی (CNN) و تأثیر الگوریتم‌های بهینه‌ساز روی دیتاست MNIST انجام شده است. هدف اصلی، درک تفاوت دینامیک یادگیری و قدرت تعمیم‌پذیری (Generalization) در ترکیب‌های مختلف معماری و بهینه‌ساز است.

### 🔬 طراحی آزمایش
* معماری‌های پیاده‌سازی شده شامل مدل کلاسیک (LeNet-5 Classic) و مدل مدرن (LeNet-5 Modern) هستند.
* بهینه‌سازهای مورد مقایسه شامل SGD و AdamW می‌باشند.
* برای ایجاد شرایط کاملاً عادلانه و ایزوله در آزمایش، وزن‌های اولیه در تمامی ۴ مدل با استفاده از `state_dict` با یکدیگر یکسان‌سازی شده است تا نقطه شروع یکسانی داشته باشند.
* آموزش مدل‌ها با استفاده از کلاس کاستوم `Trainer` به همراه ذخیره‌سازی خودکار بهترین وزن‌ها (Checkpoints) به مدت ۵۰ اپوک انجام شد.

### 📊 نتایج نهایی ارزیابی (داده‌های تست)
بر اساس ارزیابی مدل‌ها روی داده‌های کاملاً ندیده (Test Data)، نتایج زیر به دست آمد:
* بالاترین دقت متعلق به مدل LeNet-5 Classic (SGD) با ۹۸.۸۴٪ بود.
* دقت تست برای مدل LeNet-5 Classic (AdamW) معادل ۹۸.۸۱٪ محاسبه شد.
* دقت تست برای مدل LeNet-5 Modern (AdamW) برابر با ۹۸.۶۸٪ گزارش شد.
* کمترین دقت مربوط به مدل LeNet-5 Modern (SGD) با ۹۸.۵۶٪ ثبت شد.

### 💡 نتیجه‌گیری و تحلیل‌های کلیدی
۱. **پیروزی کمینه‌های هموار (Flat Minima):** مدل کلاسیک آموزش‌دیده با SGD بهترین عملکرد کلی (۹۸.۸۴٪) را ثبت کرد. این نشان می‌دهد الگوریتم کلاسیک SGD با وجود سرعت همگرایی کمتر در اپوک‌های اولیه، در نهایت با نشستن در کمینه‌های هموارتر، قدرت تعمیم و پایداری بهتری نسبت به الگوریتم تطبیقی AdamW دارد.

۲. **برتری سوگیری استقرایی (Inductive Bias):** در نتایج به دست آمده، معماری کلاسیک به طور محسوسی از نسخه مدرن (استرایدی) بهتر عمل کرد. این موضوع ثابت می‌کند که روی تصاویر ساده‌ای مثل ارقام MNIST، لایه‌های Pooling با ایجاد ویژگی «تغییرناپذیری انتقالی» عملکرد بهتری نسبت به واگذاری وظیفه کاهش ابعاد به فیلترهای Strided Convolution دارند.

۳. **رفتار برتر AdamW در فضاهای پیچیده:** در معماری مدرن، بر خلاف معماری کلاسیک، بهینه‌ساز AdamW (با ۹۸.۶۸٪) توانست از SGD (با ۹۸.۵۶٪) پیشی بگیرد. حذف Pooling باعث ناهموارتر شدن فضای بهینه‌سازی می‌شود و در این شرایط، تنظیمات تطبیقی و مستقلِ نرخ یادگیری در الگوریتم AdamW برای همگرایی بهتر فیلترهای گام‌دار بسیار کارآمدتر از SGD عمل می‌کند.

---
*این نوت‌بوک شامل مراحل کامل پیش‌پردازش استاندارد پایتورچ، دیتالودرها، حلقه‌های آموزش و اعتبارسنجی یکپارچه می‌باشد.*
</div>

## 1. Imports & Setup


In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset ,DataLoader
import matplotlib.pyplot as plt

from models.cnn import lenet5_classic
####
from utils.trainer import Trainer
from models.cnn.lenet5_classic import LeNet5_C
from models.cnn.lenet5_modern import LeNet5_M

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## 2. Load Data & Exploratory Data Analysis (EDA)


In [3]:
df_train = pd.read_csv("../data/mnist/mnist_train.csv")
df_test = pd.read_csv("../data/mnist/mnist_test.csv")

## 3. Data Visualization


## 4. Preprocessing & DataLoaders


In [4]:
X_train = df_train.iloc[1:, 1:].values
y_train = df_train.iloc[1:, 0].values
X_test = df_test.iloc[1:, 1:].values
y_test = df_test.iloc[1:, 0].values

In [5]:
X_train = X_train.reshape(-1,1,28,28)
X_test = X_test.reshape(-1,1,28,28)

In [6]:
X_train ,X_val ,y_train ,y_val = train_test_split(X_train ,y_train ,test_size = 0.2 ,random_state = 42)

In [7]:
X_train = torch.tensor(X_train,dtype=torch.float32)/ 255.0
y_train = torch.tensor(y_train,dtype=torch.int64)

X_val = torch.tensor(X_val,dtype=torch.float32)/ 255.0
y_val = torch.tensor(y_val,dtype=torch.int64)

X_test = torch.tensor(X_test,dtype=torch.float32)/ 255.0
y_test = torch.tensor(y_test,dtype=torch.int64)

In [8]:
tensor_train = TensorDataset(X_train,y_train)
tensor_val = TensorDataset(X_val,y_val)
tensor_test = TensorDataset(X_test,y_test)

In [9]:
train_dataloader = DataLoader(tensor_train,batch_size=256,shuffle=True)
test_dataloader = DataLoader(tensor_test,batch_size=256,shuffle=False)
val_dataloader = DataLoader(tensor_val,batch_size=256,shuffle=False)

## 5. Model Architecture


In [10]:
lenet_classic_sgd = LeNet5_C().to(device)
lenet_classic_adamw = LeNet5_C().to(device)
lenet_modern_sgd = LeNet5_M().to(device)
lenet_modern_adamw = LeNet5_M().to(device)


In [11]:
list_model = [lenet_classic_sgd, lenet_modern_sgd]

for idx, model in enumerate(list_model, start=1):
    total_parameters = 0
    for  p in model.parameters():
        if p.requires_grad:
            total_parameters += p.numel()

    print(f"model {idx} has {total_parameters} trainable parameters")

model 1 has 61706 trainable parameters
model 2 has 61706 trainable parameters


## 6. Loss Function & Optimizer


In [12]:
criterion = nn.CrossEntropyLoss()

optimizer_lenet_classic_sgd = torch.optim.SGD(lenet_classic_sgd.parameters(), lr = 0.01, momentum = 0.9 , weight_decay = 0.0005)
optimizer_lenet_classic_adamw = torch.optim.AdamW(lenet_classic_adamw.parameters(),lr = 0.001,weight_decay = 0.0005)
optimizer_lenet_modern_sgd = torch.optim.SGD(lenet_modern_sgd.parameters(), lr = 0.01, momentum = 0.9 , weight_decay = 0.0005)
optimizer_lenet_modern_adamw = torch.optim.AdamW(lenet_modern_adamw.parameters(),lr = 0.001,weight_decay = 0.0005)

## 7. Training & Evaluation Pipeline


In [14]:
# ۱. هم‌وزن کردن کلاسیک‌ها
lenet_classic_adamw.load_state_dict(lenet_classic_sgd.state_dict())

# ۲. انتقال وزن‌های کلاسیک به مدرن (با تطبیق ایندکس ۳ به ۲)
lenet_modern_sgd.feature_extractor[0].load_state_dict(lenet_classic_sgd.feature_extractor[0].state_dict())
lenet_modern_sgd.feature_extractor[2].load_state_dict(lenet_classic_sgd.feature_extractor[3].state_dict())
lenet_modern_sgd.classifier.load_state_dict(lenet_classic_sgd.classifier.state_dict())

# ۳. هم‌وزن کردن مدرن دوم با مدرن اول
lenet_modern_adamw.load_state_dict(lenet_modern_sgd.state_dict())

<All keys matched successfully>

In [15]:
tr_va_classic_sgd = Trainer(lenet_classic_sgd,optimizer_lenet_classic_sgd,criterion,device)
hestory_tr_va_classic_sgd = tr_va_classic_sgd.fit(train_dataloader,val_dataloader,50,model_name="classic_sgd")

Starting training on device: cuda for 50 epochs...

Epoch [01/50] | Train Loss: 1.4706 - Train Acc: 57.59% | Val Loss: 0.5313 - Val Acc: 84.92% --> (New Best! Saved to ../checkpoints/classic_sgd.pth)
Epoch [02/50] | Train Loss: 0.3878 - Train Acc: 89.18% | Val Loss: 0.3220 - Val Acc: 90.33% --> (New Best! Saved to ../checkpoints/classic_sgd.pth)
Epoch [03/50] | Train Loss: 0.2767 - Train Acc: 92.04% | Val Loss: 0.2526 - Val Acc: 92.19% --> (New Best! Saved to ../checkpoints/classic_sgd.pth)
Epoch [04/50] | Train Loss: 0.2186 - Train Acc: 93.64% | Val Loss: 0.2024 - Val Acc: 93.92% --> (New Best! Saved to ../checkpoints/classic_sgd.pth)
Epoch [05/50] | Train Loss: 0.1774 - Train Acc: 94.77% | Val Loss: 0.1722 - Val Acc: 94.74% --> (New Best! Saved to ../checkpoints/classic_sgd.pth)
Epoch [06/50] | Train Loss: 0.1476 - Train Acc: 95.71% | Val Loss: 0.1434 - Val Acc: 95.73% --> (New Best! Saved to ../checkpoints/classic_sgd.pth)
Epoch [07/50] | Train Loss: 0.1264 - Train Acc: 96.32% | Val

In [16]:
tr_va_classic_adamw = Trainer(lenet_classic_adamw,optimizer_lenet_classic_adamw,criterion,device)
hestory_tr_va_classic_adamw = tr_va_classic_adamw.fit(train_dataloader,val_dataloader,50,"classic_adamw")

Starting training on device: cuda for 50 epochs...

Epoch [01/50] | Train Loss: 0.5993 - Train Acc: 83.54% | Val Loss: 0.2547 - Val Acc: 92.17% --> (New Best! Saved to ../checkpoints/classic_adamw.pth)
Epoch [02/50] | Train Loss: 0.1932 - Train Acc: 94.22% | Val Loss: 0.1711 - Val Acc: 94.49% --> (New Best! Saved to ../checkpoints/classic_adamw.pth)
Epoch [03/50] | Train Loss: 0.1261 - Train Acc: 96.27% | Val Loss: 0.1180 - Val Acc: 96.34% --> (New Best! Saved to ../checkpoints/classic_adamw.pth)
Epoch [04/50] | Train Loss: 0.0923 - Train Acc: 97.17% | Val Loss: 0.0932 - Val Acc: 97.19% --> (New Best! Saved to ../checkpoints/classic_adamw.pth)
Epoch [05/50] | Train Loss: 0.0727 - Train Acc: 97.84% | Val Loss: 0.0808 - Val Acc: 97.53% --> (New Best! Saved to ../checkpoints/classic_adamw.pth)
Epoch [06/50] | Train Loss: 0.0589 - Train Acc: 98.21% | Val Loss: 0.0720 - Val Acc: 97.81% --> (New Best! Saved to ../checkpoints/classic_adamw.pth)
Epoch [07/50] | Train Loss: 0.0485 - Train Acc: 

In [17]:
tr_va_modern_sgd = Trainer(lenet_modern_sgd,optimizer_lenet_modern_sgd,criterion,device)
hestory_tr_va_modern_sgd = tr_va_modern_sgd.fit(train_dataloader,val_dataloader,50,"modern_sgd")

Starting training on device: cuda for 50 epochs...

Epoch [01/50] | Train Loss: 2.2982 - Train Acc: 10.95% | Val Loss: 2.2819 - Val Acc: 14.31% --> (New Best! Saved to ../checkpoints/modern_sgd.pth)
Epoch [02/50] | Train Loss: 1.1490 - Train Acc: 66.98% | Val Loss: 0.3891 - Val Acc: 88.17% --> (New Best! Saved to ../checkpoints/modern_sgd.pth)
Epoch [03/50] | Train Loss: 0.2889 - Train Acc: 91.22% | Val Loss: 0.2315 - Val Acc: 92.71% --> (New Best! Saved to ../checkpoints/modern_sgd.pth)
Epoch [04/50] | Train Loss: 0.1933 - Train Acc: 94.06% | Val Loss: 0.1690 - Val Acc: 94.70% --> (New Best! Saved to ../checkpoints/modern_sgd.pth)
Epoch [05/50] | Train Loss: 0.1453 - Train Acc: 95.56% | Val Loss: 0.1533 - Val Acc: 95.24% --> (New Best! Saved to ../checkpoints/modern_sgd.pth)
Epoch [06/50] | Train Loss: 0.1227 - Train Acc: 96.19% | Val Loss: 0.1264 - Val Acc: 96.17% --> (New Best! Saved to ../checkpoints/modern_sgd.pth)
Epoch [07/50] | Train Loss: 0.1005 - Train Acc: 96.84% | Val Loss:

In [18]:
tr_va_modern_adamw = Trainer(lenet_modern_adamw,optimizer_lenet_modern_adamw,criterion,device)
hestory_tr_va_modern_adamw = tr_va_modern_adamw.fit(train_dataloader,val_dataloader,50,"modern_adamw")

Starting training on device: cuda for 50 epochs...

Epoch [01/50] | Train Loss: 0.7600 - Train Acc: 76.24% | Val Loss: 0.2830 - Val Acc: 91.69% --> (New Best! Saved to ../checkpoints/modern_adamw.pth)
Epoch [02/50] | Train Loss: 0.2120 - Train Acc: 93.86% | Val Loss: 0.1732 - Val Acc: 94.92% --> (New Best! Saved to ../checkpoints/modern_adamw.pth)
Epoch [03/50] | Train Loss: 0.1332 - Train Acc: 96.11% | Val Loss: 0.1338 - Val Acc: 95.93% --> (New Best! Saved to ../checkpoints/modern_adamw.pth)
Epoch [04/50] | Train Loss: 0.0972 - Train Acc: 97.15% | Val Loss: 0.1021 - Val Acc: 96.89% --> (New Best! Saved to ../checkpoints/modern_adamw.pth)
Epoch [05/50] | Train Loss: 0.0774 - Train Acc: 97.71% | Val Loss: 0.0889 - Val Acc: 97.25% --> (New Best! Saved to ../checkpoints/modern_adamw.pth)
Epoch [06/50] | Train Loss: 0.0680 - Train Acc: 97.90% | Val Loss: 0.0824 - Val Acc: 97.53% --> (New Best! Saved to ../checkpoints/modern_adamw.pth)
Epoch [07/50] | Train Loss: 0.0566 - Train Acc: 98.21%

In [19]:
def evaluate_checkpoint(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return (correct / total) * 100.0


# ۲. مشخصات مدل‌ها و مسیر فایل‌های ذخیره‌شده
models_config = [
    {"name": "LeNet-5 Classic (SGD)",   "class": LeNet5_C, "ckpt": "../checkpoints/classic_sgd.pth"},
    {"name": "LeNet-5 Classic (AdamW)", "class": LeNet5_C, "ckpt": "../checkpoints/classic_adamw.pth"},
    {"name": "LeNet-5 Modern (SGD)",    "class": LeNet5_M, "ckpt": "../checkpoints/modern_sgd.pth"},
    {"name": "LeNet-5 Modern (AdamW)",  "class": LeNet5_M, "ckpt": "../checkpoints/modern_adamw.pth"}
]

# ۳. اجرای ارزیابی برای هر مدل
results = []

for config in models_config:
    # ساخت یک نمونه جدید از معماری و ارسال به پردازنده
    model = config["class"]().to(device)

    # لود کردن بهترین وزن‌های ذخیره‌شده
    model.load_state_dict(torch.load(config["ckpt"], map_location=device))

    # محاسبه دقت روی داده‌های تست
    test_accuracy = evaluate_checkpoint(model, test_dataloader, device)

    results.append({
        "Model Architecture": config["name"],
        "Test Accuracy (%)": f"{test_accuracy:.2f}%"
    })

# ۴. نمایش خروجی نهایی
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

     Model Architecture Test Accuracy (%)
  LeNet-5 Classic (SGD)            98.84%
LeNet-5 Classic (AdamW)            98.81%
   LeNet-5 Modern (SGD)            98.56%
 LeNet-5 Modern (AdamW)            98.68%
